# Reproduction des résultats — SU(2) × Sim(2) sur MNIST

**Notebook de vérification pour reviewer.** Aucune architecture n'est définie ici,
aucun entraînement n'est relancé. Les modèles sont chargés depuis le dépôt sous
forme sérialisée (TorchScript) et le notebook rejoue **l'intégralité du protocole
d'évaluation de l'article**.

---

## Ce qui est reproduit

| Phase | Contenu | Sortie |
|---|---|---|
| **A** | Accuracy propre, nombre de paramètres, latence | `table1_clean_accuracy.csv` |
| **B** | Équivariance architecturale (poids aléatoires **et** entraînés) | `equivariance_*.csv/png` |
| **C** | Planches visuelles des transformations appliquées | `transforms_*.png` |
| **D** | Sweeps de robustesse φ / ψ₁ / échelle / projectif / combo / Haar SU(2) | `rob_sweep_*.csv`, `robustness_curves.png` |
| **E** | Courbes d'apprentissage (relues, non recalculées) | `training_curves.png` |
| **F** | Tableau de synthèse + archive `.zip` | `reviewer_summary.csv` |

## Ce qui n'est **pas** dans ce notebook

- les définitions de modèles (`ClassicResNet`, `Sim2OnlyResNet`, `SU2SphericalResNetSO3`,
  `BiLogPolarMöbiusResNet`) ;
- la boucle d'entraînement, l'optimiseur, le scheduler ;
- le jeu d'entraînement MNIST — seul le jeu de test est téléchargé.

## Comment lancer

1. Exécuter la cellule de configuration (renseigner `REPO_URL`).
2. `Exécution → Tout exécuter`.
3. Laisser `RUN_MODE = "quick"` pour une première vérification (~5 min sur T4),
   puis `"full"` pour le protocole exact de l'article (~35–50 min).

> **Note d'intégrité.** Chaque poids est vérifié par SHA-256 contre le manifeste
> livré avec le dépôt : le notebook signale toute divergence entre les fichiers
> chargés et ceux déclarés par les auteurs.

---

## 0 · Configuration

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION  —  seule cellule à modifier
# ══════════════════════════════════════════════════════════════════════════════

# ── Dépôt contenant les poids pré-entraînés ──────────────────────────────────
REPO_URL   = "https://github.com/<ORG>/<REPO>.git"   # ← à remplacer
REPO_DIR   = "su2_sim2_repro"
CKPT_SUBDIR = "checkpoints/mnist_v10"                # dossier des poids dans le dépôt

# ── Sorties ──────────────────────────────────────────────────────────────────
OUT_DIR = "./reviewer_outputs"

# ── Régime d'exécution ───────────────────────────────────────────────────────
#   "full"  : protocole exact de l'article (10 000 images de test, tous les
#             points de sweep).                      GPU T4 ≈ 35–50 min
#   "quick" : sous-échantillonnage, ~8× plus rapide, mêmes tendances.
#             À utiliser pour une première vérification.   ≈ 5 min
RUN_MODE = "quick"        # "quick" | "full"

# ── Protocole d'évaluation ───────────────────────────────────────────────────
#   "paper"         : transformation de validation identique à l'entraînement
#                     (Resize 64 + RandomCrop 64 pad 6), graine fixée → résultats
#                     reproductibles au bit près.
#   "deterministic" : Resize 64 + padding réfléchi + CenterCrop, aucun aléa.
#                     Donne des chiffres très légèrement différents.
EVAL_TRANSFORM = "paper"

SEED = 42

# ── Classes exclues des sweeps (ambiguïté 6↔9 sous rotation) ─────────────────
EXCLUDED_CLASSES = (1, 9)

# ── Régimes ──────────────────────────────────────────────────────────────────
PRESETS = {
    "full":  dict(n_val=None, n_phi=20, n_psi1=20, n_proj=40, n_combo=20, n_su2=32,
                  batch_size=128),
    "quick": dict(n_val=2000, n_phi=10, n_psi1=10, n_proj=12, n_combo=10, n_su2=12,
                  batch_size=256),
}
CFG = PRESETS[RUN_MODE]
print(f"⚙️  Mode « {RUN_MODE} » → {CFG}")

## 1 · Installation et récupération des poids

Aucune dépendance géométrique n'est nécessaire : les modèles sont autoportants.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  INSTALLATION & RÉCUPÉRATION DES POIDS
# ══════════════════════════════════════════════════════════════════════════════
import os, subprocess, sys

# torch, torchvision, numpy, pandas et matplotlib sont préinstallés sur Colab.
#
# torch-harmonics est requis pour reconstruire l'architecture SU2Stereo.
# ⚠️  Les wheels 0.9.x ne couvrent que cp310–cp312 et il n'existe pas de
#     distribution source. Sous Python 3.13, pip redescend silencieusement à la
#     0.8.0, dont les transformées sphériques diffèrent. On compile donc depuis
#     les sources au bon tag si le runtime est en 3.13.
if sys.version_info[:2] <= (3, 12):
    subprocess.run(["pip", "install", "-q", "torch-harmonics==0.9.1"], check=True)
else:
    print("⚙️  Python 3.13 détecté → compilation de torch-harmonics 0.9.1 "
          "depuis les sources (quelques minutes)…")
    subprocess.run(["pip", "install", "-q", "--no-build-isolation",
                    "git+https://github.com/NVIDIA/torch-harmonics.git@v0.9.1"],
                   check=True)

import torch_harmonics
print(f"✅ torch-harmonics {torch_harmonics.__version__} | Python "
      f"{sys.version.split()[0]}")

if not os.path.isdir(REPO_DIR):
    print(f"⬇️  Clonage de {REPO_URL} …")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    print(f"✅ Dépôt déjà présent : {REPO_DIR}")

ckpt_dir = os.path.join(REPO_DIR, CKPT_SUBDIR)
print(f"\n📂 {ckpt_dir}")
if os.path.isdir(ckpt_dir):
    for f in sorted(os.listdir(ckpt_dir)):
        size = os.path.getsize(os.path.join(ckpt_dir, f)) / 1e6
        print(f"   {f:<32} {size:8.2f} Mo")
else:
    print("   ❌ dossier introuvable — vérifiez CKPT_SUBDIR.")

## 2 · Imports et constantes

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  IMPORTS & CONSTANTES
#  (aucune définition de modèle ici — ce notebook ne fait que de l'inférence)
# ══════════════════════════════════════════════════════════════════════════════
import math, os, gc, json, time, hashlib, random, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm
from IPython.display import display, Image as IPImage

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
torch.backends.cudnn.benchmark = True

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything()
os.makedirs(OUT_DIR, exist_ok=True)

def show_png(path, width=1100):
    """Affiche une figure déjà écrite sur disque (matplotlib est en backend Agg)."""
    if os.path.exists(path):
        display(IPImage(filename=path, width=width))
    else:
        print(f"⚠️  figure absente : {path}")

# ── Palette / libellés (identiques à ceux de l'article) ──────────────────────
MODEL_COLORS  = {"Classic": "#e15759", "Sim2Only": "#4e79a7",
                 "SU2Stereo": "#59a14f", "BiLogPolar": "#f28e2b"}
MODEL_MARKERS = {"Classic": "s", "Sim2Only": "D",
                 "SU2Stereo": "^", "BiLogPolar": "o"}
MODEL_LABELS  = {"Classic": "Classique", "Sim2Only": "Sim(2)",
                 "SU2Stereo": "SU(2) Stéréo", "BiLogPolar": "Bi-LogPolar"}

print(f"🖥️  device        : {device}  "
      f"({torch.cuda.get_device_name(0) if device.type=='cuda' else 'CPU'})")
print(f"🔥 torch          : {torch.__version__}")
print(f"📁 sorties        : {os.path.abspath(OUT_DIR)}")

## 3 · Boîte à outils d'évaluation

Ces quatre cellules sont reprises **verbatim** du notebook d'entraînement, à une exception près, signalée dans le code : l'extraction des descripteurs passe par le contrat `forward(x) → (logits, features)` au lieu d'un hook, puisque les modèles sont ici des boîtes noires.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  TRANSFORMATIONS & GRILLES DE SWEEP
#  Bloc repris VERBATIM du notebook d'entraînement : c'est la garantie que le
#  reviewer évalue sur exactement les mêmes transformations que l'article.
# ══════════════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════════════════
# TRANSFORMATIONS POUR LES TESTS D'ÉQUIVARIANCE ET DE ROBUSTESSE
# ══════════════════════════════════════════════════════════════════════════════

def apply_rotation_2d(x, deg):
    t = math.radians(deg)
    aff = torch.tensor([[math.cos(t), -math.sin(t), 0.],
                         [math.sin(t),  math.cos(t), 0.]], device=x.device)
    aff = aff.unsqueeze(0).expand(x.shape[0], -1, -1)
    return F.grid_sample(x, F.affine_grid(aff, x.shape, align_corners=True),
                         align_corners=True, mode='bilinear', padding_mode='zeros')


def apply_mobius_phi(x, beta):
    """Möbius SU(2) radiale : z → (z+t)/(1-tz), t = tan(β/2)."""
    t = math.tan(float(beta) / 2)
    lin = torch.linspace(-1, 1, x.shape[-1], device=x.device)
    gr = torch.stack(torch.meshgrid(lin, lin, indexing='xy'))
    z = torch.complex(gr[0], gr[1])
    zp = (z + t) / (1 - t * z)
    grid = torch.stack([zp.real.clamp(-1, 1), zp.imag.clamp(-1, 1)], dim=-1)
    return F.grid_sample(x, grid.unsqueeze(0).expand(x.shape[0], -1, -1, -1),
                         mode='bilinear', padding_mode='border', align_corners=True)


def apply_scale(x, scale):
    s = 1. / scale
    aff = torch.tensor([[s, 0., 0.], [0., s, 0.]], device=x.device)
    aff = aff.unsqueeze(0).expand(x.shape[0], -1, -1)
    return F.grid_sample(x, F.affine_grid(aff, x.shape, align_corners=True),
                         align_corners=True, mode='bilinear', padding_mode='zeros')


def apply_projective(x, c1=0.001, c2=0.001):
    H_mat = np.array([[1., 0., 0.], [0., 1., 0.], [c1, c2, 1.]])
    try:    H_inv = np.linalg.inv(H_mat)
    except: H_inv = np.eye(3)
    lin = torch.linspace(-1, 1, x.shape[-1], device=x.device)
    yy, xx = torch.meshgrid(lin, lin, indexing='ij')
    pts = torch.stack([xx.flatten(), yy.flatten(),
                       torch.ones(x.shape[-1] ** 2, device=x.device)])
    H_t = torch.tensor(H_inv, dtype=torch.float32, device=x.device)
    src = H_t @ pts
    w_ = src[2:3].clamp(min=1e-8)
    grid = (src[:2] / w_).T.reshape(x.shape[-1], x.shape[-1], 2)
    return F.grid_sample(x, grid.unsqueeze(0).expand(x.shape[0], -1, -1, -1),
                         mode='bilinear', padding_mode='zeros', align_corners=True)


def make_radial_images(n, size, device):
    lin = torch.linspace(-1, 1, size, device=device)
    yy, xx = torch.meshgrid(lin, lin, indexing='ij')
    r = (xx**2 + yy**2).sqrt()
    img = r.unsqueeze(0).unsqueeze(0).expand(n, 1, size, size)  # ← (N, 1, H, W)
    return img.contiguous()


# ══════════════════════════════════════════════════════════════════════════════
# GRILLES DE SWEEP (robustesse)
# ══════════════════════════════════════════════════════════════════════════════

def _mobius_grid(mat, grid_r2):
    a, b = mat[0, 0], mat[0, 1]
    z = torch.complex(grid_r2[0], grid_r2[1])
    denom = -torch.conj(b) * z + torch.conj(a)
    zp = (a * z + b) / (denom + 1e-8)
    return torch.stack([zp.real, zp.imag], dim=-1)


def _build_phi_grids_manual(phi_values, grid_r2):
    grids = []
    for phi in phi_values:
        alpha = torch.tensor(math.cos(phi / 2), dtype=torch.complex64)
        beta = torch.tensor(1j * math.sin(phi / 2), dtype=torch.complex64)
        mat = torch.tensor([[alpha, beta], [-beta.conj(), alpha.conj()]],
                           dtype=torch.complex64, device=grid_r2.device)
        a, b = mat[0, 0].conj(), mat[1, 0].conj()
        z = torch.complex(grid_r2[0], grid_r2[1])
        denom = -b.conj() * z + a.conj()
        zp = (a * z + b) / (denom + 1e-8)
        grids.append(torch.stack([zp.real, zp.imag], dim=-1))
    return torch.stack(grids)


def _build_psi1_grids(n_steps, grid_r2, phi_fixed=0., max_deg=90.):
    max_rad = math.radians(max_deg)
    psi1_vals = torch.linspace(0., max_rad, n_steps)
    phi = torch.full((n_steps,), phi_fixed)
    psi2 = torch.zeros(n_steps)
    alpha = torch.cos(phi / 2) * torch.complex(torch.cos((psi1_vals + psi2) / 2),
                                                torch.sin((psi1_vals + psi2) / 2))
    beta = 1j * torch.sin(phi / 2) * torch.complex(torch.cos((psi1_vals - psi2) / 2),
                                                    torch.sin((psi1_vals - psi2) / 2))
    mats = torch.stack([torch.stack([torch.stack([alpha, beta], -1),
                                     torch.stack([-torch.conj(beta), torch.conj(alpha)], -1)], -2)])[0]
    mats = mats.to(grid_r2.device)
    grids = torch.stack([_mobius_grid(mats[i].mH, grid_r2) for i in range(n_steps)])
    return psi1_vals.numpy(), np.degrees(psi1_vals.numpy()), grids


def _build_scale_grids(scale_values, H, W, device):
    grids = []
    for s in scale_values:
        xs = torch.linspace(-1. / s, 1. / s, W, device=device)
        ys = torch.linspace(-1. / s, 1. / s, H, device=device)
        yy, xx = torch.meshgrid(ys, xs, indexing='ij')
        grids.append(torch.stack([xx, yy], dim=-1))
    return torch.stack(grids), np.array(scale_values, dtype=np.float32)


def _build_projective_grids(n_samples, H, W, device, seed=42,
                             mnist_safe=False):          # ← NOUVEAU flag
    rng = np.random.default_rng(seed)

    if mnist_safe:
        # ── Amplitudes réduites : préserve la classe du chiffre ───────────────
        thetas = rng.uniform(-math.pi / 12, math.pi / 12, n_samples)  # ±15°  (était ±180°)
        scales = rng.uniform(0.85, 1.15, n_samples)                    # ±15%  (était 0.3–1.7)
        shears = rng.uniform(-10., 10., n_samples)                     # ±10°  (était ±45°)
        txs    = rng.uniform(-0.05, 0.05, n_samples)                   # ±5%   (était ±10%)
        tys    = rng.uniform(-0.05, 0.05, n_samples)                   # ±5%
        c1s    = rng.uniform(-0.001, 0.001, n_samples)                 # ±0.001 (était ±0.002)
        c2s    = rng.uniform(-0.001, 0.001, n_samples)
    else:
        # ── Amplitudes originales (STL-10) ────────────────────────────────────
        thetas = rng.uniform(-math.pi, math.pi, n_samples)
        scales = rng.uniform(0.3, 1.7, n_samples)
        shears = rng.uniform(-45., 45., n_samples)
        txs    = rng.uniform(-0.1, 0.1, n_samples)
        tys    = rng.uniform(-0.1, 0.1, n_samples)
        c1s    = rng.uniform(-0.002, 0.002, n_samples)
        c2s    = rng.uniform(-0.002, 0.002, n_samples)

    xs = torch.linspace(-1, 1, W, device=device)
    ys = torch.linspace(-1, 1, H, device=device)
    yy, xx  = torch.meshgrid(ys, xs, indexing='ij')
    pts_out = torch.stack([xx.flatten(), yy.flatten(),
                           torch.ones(H * W, device=device)])
    grids = []
    for i in range(n_samples):
        ct, st = math.cos(thetas[i]), math.sin(thetas[i])
        sh = math.tan(math.radians(shears[i]))
        R  = np.array([[ct, -st, 0.], [st,  ct, 0.], [0., 0., 1.]])
        Sh = np.array([[1., sh,  0.], [0.,  1., 0.], [0., 0., 1.]])
        S  = np.diag([scales[i], scales[i], 1.])
        P  = np.array([[1., 0., 0.], [0., 1., 0.], [c1s[i], c2s[i], 1.]])
        T  = np.array([[1., 0., txs[i]], [0., 1., tys[i]], [0., 0., 1.]])
        H_mat = T @ P @ S @ Sh @ R
        try:    H_inv = np.linalg.inv(H_mat)
        except: H_inv = np.eye(3)
        H_t = torch.tensor(H_inv, dtype=torch.float32, device=device)
        src = H_t @ pts_out
        w_  = src[2:3].clamp(min=1e-8)
        grids.append((src[:2] / w_).T.reshape(H, W, 2))
    return torch.stack(grids)

# ════════════════════════════════════════════════════════════════════════════
# HELPER : _build_combo_grids
# Construit les grilles pour la transformation Möbius combinée φ+ψ₁
# ════════════════════════════════════════════════════════════════════════════

def _build_combo_grids(phi_values, psi1_values, grid_r2):
    """
    Compose Möbius-φ et Möbius-ψ₁ en une seule transformation :
    multiplie les matrices SU(2) et applique _mobius_grid une fois.

    phi_values  : array (N,) en radians  — paramètre de dilatation
    psi1_values : array (N,) en radians  — paramètre de rotation S¹
    grid_r2     : tensor (2, H, W)       — grille identité en [-1,1]

    Retourne : tensor (N, H, W, 2)
    """
    grids = []

    for phi, psi1 in zip(phi_values, psi1_values):

        # ── Matrice SU(2) pour φ (dilatation / Möbius réel) ──────────────────
        alpha_phi = torch.tensor(math.cos(phi / 2), dtype=torch.complex64)
        beta_phi  = torch.tensor(1j * math.sin(phi / 2), dtype=torch.complex64)
        M_phi = torch.tensor(
            [[alpha_phi,            beta_phi],
             [-beta_phi.conj(), alpha_phi.conj()]],
            dtype=torch.complex64, device=grid_r2.device
        )

        # ── Matrice SU(2) pour ψ₁ (rotation sur S¹, φ_fixed=0) ──────────────
        # phi_fixed=0 → alpha = exp(i ψ₁/2), beta = 0
        alpha_psi = torch.complex(
            torch.tensor(math.cos(psi1 / 2)),
            torch.tensor(math.sin(psi1 / 2))
        ).to(torch.complex64).to(grid_r2.device)
        beta_psi = torch.zeros(1, dtype=torch.complex64, device=grid_r2.device).squeeze()
        M_psi = torch.tensor(
            [[alpha_psi,            beta_psi],
             [-beta_psi.conj(), alpha_psi.conj()]],
            dtype=torch.complex64, device=grid_r2.device
        )

        # ── Composition : M_combo = M_psi @ M_phi ────────────────────────────
        # Ordre : φ d'abord, puis ψ₁ (comme apply_mobius_psi1(apply_mobius_phi(x)))
        M_combo = M_psi @ M_phi

        # ── _mobius_grid attend mat.mH  (conjuguée-transposée) ───────────────
        # (même convention que _build_psi1_grids)
        grids.append(_mobius_grid(M_combo.mH, grid_r2))

    return torch.stack(grids)   # (N, H, W, 2)

def _build_su2_random_grids(n_samples, grid_r2, seed=42,
                             max_phi  = math.pi / 4,   # 45° — φ peut être plus grand, pas de flip
                             max_psi1 = math.pi / 6,   # 30° — ψ₁ limité, c'est lui qui retourne
                             max_psi2 = math.pi / 4):  # 45°
    """
    Tire n_samples transformations SU(2) selon la mesure de Haar
    avec angles restreints pour MNIST (évite 6↔9).

    φ  ∈ [0, π/4]  — dilatation Möbius       (pas de flip)
    ψ₁ ∈ [0, π/6]  — rotation S¹ ±30°        (cause principale du 6→9)
    ψ₂ ∈ [0, π/4]  — rotation conjuguée ±45°
    """
    rng = np.random.default_rng(seed)

    # Haar sur [0, max_phi] : p(φ) ∝ sin(φ)
    u         = rng.uniform(0, 1, n_samples)
    phi_vals  = np.arccos(1 - u * (1 - math.cos(max_phi)))
    psi1_vals = rng.uniform(0, max_psi1, n_samples)
    psi2_vals = rng.uniform(0, max_psi2, n_samples)

    grids = []
    for phi, psi1, psi2 in zip(phi_vals, psi1_vals, psi2_vals):
        alpha = torch.tensor(
            math.cos(phi / 2) * complex(math.cos((psi1 + psi2) / 2),
                                         math.sin((psi1 + psi2) / 2)),
            dtype=torch.complex64
        ).to(grid_r2.device)
        beta = torch.tensor(
            1j * math.sin(phi / 2) * complex(math.cos((psi1 - psi2) / 2),
                                              math.sin((psi1 - psi2) / 2)),
            dtype=torch.complex64
        ).to(grid_r2.device)
        M = torch.tensor(
            [[alpha,        beta],
             [-beta.conj(), alpha.conj()]],
            dtype=torch.complex64, device=grid_r2.device
        )
        grids.append(_mobius_grid(M.mH, grid_r2))

    return torch.stack(grids), phi_vals, psi1_vals, psi2_vals

def apply_mobius_combo(x, phi, psi1):
    """
    Applique la transformation Möbius composée (φ puis ψ₁) à un batch x (B,C,H,W).
    Réutilise _build_combo_grids sur une grille construite à la volée.
    """
    H, W = x.shape[-2], x.shape[-1]
    lin     = torch.linspace(-1, 1, W, device=x.device)  # carré : W=H attendu
    lin_h   = torch.linspace(-1, 1, H, device=x.device)
    xx, yy  = torch.meshgrid(lin, lin_h, indexing='xy')
    grid_r2 = torch.stack([xx, yy])                       # (2, H, W)

    g = _build_combo_grids([phi], [psi1], grid_r2)        # (1, H, W, 2)
    g = g.expand(x.size(0), -1, -1, -1)                   # (B, H, W, 2)
    return F.grid_sample(x, g, align_corners=True,
                         mode='bilinear', padding_mode='border')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  ÉQUIVARIANCE  —  erreur cosinus entre descripteurs
#
#      err(T) = 1 − cos( feat(x) , feat(T·x) )        exprimée en %
#
#  Identique à la version de l'article ; seule l'extraction du descripteur
#  change : elle passe par ModelBundle.features() au lieu d'un hook sur le
#  pooling, puisque les modèles sont ici des boîtes noires sérialisées.
# ══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def measure_equivariance(models_dict, x_default, transforms_dict, x_per_family=None):
    if x_per_family is None:
        x_per_family = {}
    results = {tag: {} for tag in models_dict}

    for tag, model in models_dict.items():
        model.eval()
        for t_name, t_list in transforms_dict.items():
            x = x_per_family.get(t_name, x_default)
            f_ref = model.features(x)
            errs = []
            for label, tfn in t_list:
                f_t = model.features(tfn(x))
                f_ref_n = F.normalize(f_ref.float(), dim=1)
                f_t_n   = F.normalize(f_t.float(),   dim=1)
                cos_err = (1.0 - (f_ref_n * f_t_n).sum(dim=1)).mean().item()
                errs.append((label, cos_err * 100.0))
            results[tag][t_name] = errs
    return results


def equivariance_to_dataframe(results):
    rows = []
    for tag, fams in results.items():
        for fam, steps in fams.items():
            for label, err in steps:
                rows.append({'Model': MODEL_LABELS.get(tag, tag),
                             'Family': fam, 'Step': label, 'CosErr_pct': err})
    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════════
#  AFFICHAGE DE L'ÉQUIVARIANCE  (verbatim)
# ══════════════════════════════════════════════════════════════════════════════

def print_equivariance_table(results):
    tags = list(results.keys())
    W = 80
    print(f"\n{'═'*W}")
    print(f"  ÉQUIVARIANCE ARCHITECTURALE")
    print(f"  err cosine (< 10% = bon)")
    print(f"{'═'*W}")
    for t_name in results[tags[0]]:
        steps = results[tags[0]][t_name]
        print(f"\n  ── {t_name.upper()} ──")
        header = f"  {'Transformation':<20}" + "".join(
            f"  {MODEL_LABELS.get(t, t):>12}" for t in tags)
        print(header)
        print("  " + "-" * (W - 4))
        for i, (label, _) in enumerate(steps):
            row = f"  {label:<20}"
            for tag in tags:
                err = results[tag][t_name][i][1]
                flag = "✅" if err < 15 else ("⚠️ " if err < 35 else "❌")
                row += f"  {err:>8.1f}% {flag}"
            print(row)
        row = f"  {'Moyenne':<20}"
        for tag in tags:
            mean_err = np.mean([e for _, e in results[tag][t_name]])
            row += f"  {mean_err:>9.1f}%   "
        print(row)
    print(f"\n{'═'*W}")


def plot_equivariance(results, save_dir, suffix=""):
    tags = list(results.keys())
    t_names = list(results[tags[0]].keys())
    n_types = len(t_names)
    fig, axes = plt.subplots(1, n_types, figsize=(5 * n_types, 5))
    if n_types == 1: axes = [axes]
    fig.suptitle("Équivariance architecturale\nerr cosine = ||feat(T(x))-feat(x)||",
                 fontsize=12, fontweight='bold')
    for ax, t_name in zip(axes, t_names):
        n_steps = len(results[tags[0]][t_name])
        x_pos = np.arange(n_steps)
        bw = 0.8 / len(tags)
        xlabels = [label for label, _ in results[tags[0]][t_name]]
        for ti, tag in enumerate(tags):
            errs = [e for _, e in results[tag][t_name]]
            ax.bar(x_pos + ti * bw - 0.4 + bw / 2, errs, bw * 0.9,
                   color=MODEL_COLORS.get(tag, 'gray'), alpha=0.85,
                   label=MODEL_LABELS.get(tag, tag), edgecolor='white', lw=0.5)
        ax.axhline(15., color='green', ls='--', lw=1.4, alpha=0.7, label='Seuil 15%')
        ax.axhline(40., color='red', ls='--', lw=1.4, alpha=0.7, label='Seuil 40%')
        ax.set_xticks(x_pos + 0.4 - bw / 2)
        ax.set_xticklabels(xlabels, fontsize=8, rotation=20, ha='right')
        ax.set_ylabel('Erreur (%)')
        ax.set_title(t_name, fontsize=11, fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, axis='y')
        max_e = max(max(e for _, e in results[t][t_name]) for t in tags)
        ax.set_ylim(0, max_e * 1.3 + 5)
    plt.tight_layout()
    path = os.path.join(save_dir, f"equivariance{suffix}.png")
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"💾 Équivariance → {path}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  SWEEP DE ROBUSTESSE & VISUALISATION  (verbatim, hors classes exclues)
# ══════════════════════════════════════════════════════════════════════════════
# ════════════════════════════════════════════════════════════════════════════
@torch.inference_mode()
def sweep_all_models(models_dict, val_loader, phi_grids, psi1_grids,
                     scale_grids, proj_grids,
                     combo_grids=None,
                     su2_grids=None,
                     phi_labels=None, psi1_labels=None,
                     scale_labels=None, proj_labels=None):

    # Classes ambiguës sous rotation/Möbius (définies dans la configuration)
    excluded = tuple(EXCLUDED_CLASSES)

    results = {}
    for tag, model in models_dict.items():
        model.eval()
        M   = phi_grids.shape[0]
        N   = psi1_grids.shape[0]
        NS  = scale_grids.shape[0]
        NP  = proj_grids.shape[0]
        NC  = combo_grids.shape[0] if combo_grids is not None else 0
        NSU = su2_grids.shape[0]   if su2_grids   is not None else 0

        cc_phi   = torch.zeros(M,   device=device)
        cc_psi1  = torch.zeros(N,   device=device)
        cc_scale = torch.zeros(NS,  device=device)
        cc_proj  = torch.zeros(NP,  device=device)
        cc_combo = torch.zeros(NC,  device=device) if NC  > 0 else None
        cc_su2   = torch.zeros(NSU, device=device) if NSU > 0 else None
        total    = 0

        for x, y in tqdm(val_loader, desc=f"  Sweep [{tag}]"):
            x, y = x.to(device), y.to(device)

            # ── Exclut les classes ambiguës de TOUS les sweeps ────────────────
            keep = torch.ones(y.size(0), dtype=torch.bool, device=device)
            for c in excluded:
                keep &= (y != c)
            x_k = x[keep]
            y_k = y[keep]
            Bk  = y_k.size(0)
            if Bk == 0:
                continue

            total += Bk

            with torch.amp.autocast(device_type=device.type,
                                    enabled=(device.type == 'cuda')):
                for i in range(M):
                    xt = F.grid_sample(x_k, phi_grids[i].unsqueeze(0).expand(Bk, -1, -1, -1),
                                       align_corners=True, mode='bilinear', padding_mode='border')
                    cc_phi[i] += (model(xt).argmax(1) == y_k).sum()

                for i in range(N):
                    xt = F.grid_sample(x_k, psi1_grids[i].unsqueeze(0).expand(Bk, -1, -1, -1),
                                       align_corners=True, mode='bilinear', padding_mode='border')
                    cc_psi1[i] += (model(xt).argmax(1) == y_k).sum()

                for i in range(NS):
                    xt = F.grid_sample(x_k, scale_grids[i].unsqueeze(0).expand(Bk, -1, -1, -1),
                                       align_corners=True, mode='bilinear', padding_mode='border')
                    cc_scale[i] += (model(xt).argmax(1) == y_k).sum()

                for i in range(NP):
                    xt = F.grid_sample(x_k, proj_grids[i].unsqueeze(0).expand(Bk, -1, -1, -1),
                                       align_corners=True, mode='bilinear', padding_mode='border')
                    cc_proj[i] += (model(xt).argmax(1) == y_k).sum()

                if combo_grids is not None:
                    for i in range(NC):
                        xt = F.grid_sample(x_k, combo_grids[i].unsqueeze(0).expand(Bk, -1, -1, -1),
                                           align_corners=True, mode='bilinear', padding_mode='border')
                        cc_combo[i] += (model(xt).argmax(1) == y_k).sum()

                if su2_grids is not None:
                    for i in range(NSU):
                        xt = F.grid_sample(x_k, su2_grids[i].unsqueeze(0).expand(Bk, -1, -1, -1),
                                           align_corners=True, mode='bilinear', padding_mode='border')
                        cc_su2[i] += (model(xt).argmax(1) == y_k).sum()

        phi_acc   = (100. * cc_phi   / total).cpu().numpy()
        psi1_acc  = (100. * cc_psi1  / total).cpu().numpy()
        scale_acc = (100. * cc_scale / total).cpu().numpy()
        proj_acc  = (100. * cc_proj  / total).cpu().numpy()

        entry = {'phi': phi_acc, 'psi1': psi1_acc,
                 'scale': scale_acc, 'proj': proj_acc}

        excl_str = ','.join(str(c) for c in excluded)
        log = (f"  [{tag}] ΔΦ={phi_acc[0]-phi_acc[-1]:+.2f}%"
               f" | Δψ₁={psi1_acc[0]-psi1_acc[-1]:+.2f}%"
               f" | Scale Δ={scale_acc.max()-scale_acc.min():.2f}%"
               f" | Proj {proj_acc.mean():.1f}%±{proj_acc.std():.1f}%"
               f"  (excl. {excl_str})")

        if combo_grids is not None:
            combo_acc      = (100. * cc_combo / total).cpu().numpy()
            entry['combo'] = combo_acc
            log += f" | Combo Δ={combo_acc[0]-combo_acc[-1]:+.2f}%"

        if su2_grids is not None:
            su2_acc      = (100. * cc_su2 / total).cpu().numpy()
            entry['su2'] = su2_acc
            log += (f" | SU(2) μ={su2_acc.mean():.1f}%"
                    f" σ={su2_acc.std():.1f}%"
                    f" min={su2_acc.min():.1f}%")

        print(log)
        results[tag] = entry

    return results

def visualize_transformations(val_loader, phi_grids, psi1_grids, scale_grids, proj_grids,
                               phi_labels=None, psi1_labels=None,
                               scale_labels=None, proj_labels=None,
                               combo_grids=None, combo_labels=None,
                               su2_grids=None,   su2_labels=None,    # ← NOUVEAU
                               n_samples=1, save_dir=None,
                               dataset='stl10'):
    from IPython.display import display

    if dataset == 'stl10':
        mean = torch.tensor([0.4467, 0.4398, 0.4066]).view(3, 1, 1)
        std  = torch.tensor([0.2603, 0.2566, 0.2713]).view(3, 1, 1)
    else:
        mean = torch.tensor([0.1307]).view(1, 1, 1)
        std  = torch.tensor([0.3081]).view(1, 1, 1)

    def to_display(t):
        t = t.detach().cpu().float()
        t = (t * std + mean).clamp(0, 1)
        return t.permute(1, 2, 0).numpy() if t.shape[0] == 3 else t.squeeze().numpy()

    x, y = next(iter(val_loader))
    x   = x.to(device)
    img = x[:n_samples]
    cmap = None if dataset == 'stl10' else 'gray'

    configs = [
        (phi_grids,   phi_labels,   "Rotation_phi"),
        (psi1_grids,  psi1_labels,  "Rotation_psi1"),
        (scale_grids, scale_labels, "Scale"),
        (proj_grids,  proj_labels,  "Projectif"),
    ]
    if combo_grids is not None:
        configs.append((combo_grids, combo_labels, "Mobius_combo_phi_psi1"))
    if su2_grids is not None:
        configs.append((su2_grids,   su2_labels,   "SU2_Haar"))        # ← NOUVEAU

    for grids, labels, title in configs:
        n_t    = grids.shape[0]
        n_cols = 1 + n_t
        fig, axes = plt.subplots(n_samples, n_cols,
                                 figsize=(2.5 * n_cols, 2.5 * n_samples),
                                 squeeze=False)
        fig.suptitle(title.replace("_", " "), fontsize=13, fontweight='bold')

        for s in range(n_samples):
            axes[s, 0].imshow(to_display(img[s]), cmap=cmap, vmin=0, vmax=1)
            axes[s, 0].set_title("Original" if s == 0 else "", fontsize=9)
            axes[s, 0].axis('off')

            for i in range(n_t):
                with torch.no_grad():
                    xt = F.grid_sample(
                        img[s:s+1],
                        grids[i].unsqueeze(0),
                        align_corners=True, mode='bilinear', padding_mode='zeros'
                    )
                axes[s, i+1].imshow(to_display(xt[0]), cmap=cmap, vmin=0, vmax=1)
                lbl = labels[i] if labels is not None else f"#{i}"
                axes[s, i+1].set_title(str(lbl) if s == 0 else "", fontsize=8)
                axes[s, i+1].axis('off')

        plt.tight_layout()
        if save_dir is not None:
            fpath = os.path.join(save_dir, f"transforms_{title}.png")
            plt.savefig(fpath, dpi=120, bbox_inches='tight')
        display(fig)
        plt.close(fig)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  TABLES & COURBES DE ROBUSTESSE  (verbatim)
# ══════════════════════════════════════════════════════════════════════════════
def print_robustness_table(phi_values, psi1_deg, scale_values, results,
                            phi_combo_rad=None, psi1_combo_rad=None):
    tags      = list(results.keys())
    idx_sc    = int(np.argmin(np.abs(scale_values - 1.0)))
    has_combo = phi_combo_rad is not None and 'combo' in next(iter(results.values()))
    has_su2   = 'su2' in next(iter(results.values()))

    W = 96 + (20 if has_combo else 0) + (22 if has_su2 else 0)
    print(f"\n{'═'*W}")
    title = "  ROBUSTESSE — sweep φ(Möbius) | ψ₁(rotation) | Scale | Projectif"
    if has_combo: title += " | Combo φ+ψ₁"
    if has_su2:   title += " | SU(2) Haar"
    print(title)
    print(f"{'═'*W}")

    hdr = (f"  {'Modèle':<22} {'Φ@0':>6} {'ΔΦ':>6} "
           f"{'ψ₁@0':>6} {'Δψ₁':>6} "
           f"{'Sc@1':>6} {'ΔSc':>6} "
           f"{'Prj μ':>7} {'Prj σ':>6}")
    if has_combo: hdr += f" {'Cmb@0':>7} {'ΔCmb':>7}"
    if has_su2:   hdr += f" {'SU2 μ':>7} {'SU2 σ':>6} {'SU2 min':>8}"
    print(hdr)
    print(f"  {'-'*(W-4)}")

    for tag in tags:
        pa  = results[tag]['phi']
        sa  = results[tag]['psi1']
        sca = results[tag]['scale']
        pra = results[tag]['proj']
        row = (f"  {MODEL_LABELS.get(tag, tag):<22}"
               f" {pa[0]:>6.1f}% {pa[0]-pa[-1]:>+5.1f}%"
               f" {sa[0]:>6.1f}% {sa[0]-sa[-1]:>+5.1f}%"
               f" {sca[idx_sc]:>6.1f}% {sca.max()-sca.min():>5.1f}%"
               f" {pra.mean():>6.1f}% {pra.std():>5.1f}%")
        if has_combo:
            ca   = results[tag]['combo']
            row += f" {ca[0]:>6.1f}% {ca[0]-ca[-1]:>+6.1f}%"
        if has_su2:
            su2a = results[tag]['su2']
            row += f" {su2a.mean():>6.1f}% {su2a.std():>5.1f}% {su2a.min():>7.1f}%"
        print(row)

    print(f"{'═'*W}")
    print(f"  Δψ₁ ≈ 0% ✅ = invariance Sim(2) exacte  |  ΔΦ ≈ 0% ✅ = invariance SU(2)")
    if has_combo: print(f"  ΔCmb ≈ 0% ✅ = invariance conjointe φ+ψ₁")
    if has_su2:   print(f"  SU2 μ ≈ acc@0 ✅ = invariance Haar complète (6,9 exclus)")


def plot_robustness_curves(phi_values, psi1_deg, scale_values, results, save_dir,
                            phi_combo_rad=None, psi1_combo_rad=None):
    tags      = list(results.keys())
    idx_sc    = int(np.argmin(np.abs(scale_values - 1.0)))
    has_combo = phi_combo_rad is not None and 'combo' in next(iter(results.values()))
    has_su2   = 'su2' in next(iter(results.values()))

    # ── Layout dynamique ──────────────────────────────────────────────────────
    # Ligne 0 : φ | ψ₁ | combo(opt)
    # Ligne 1 : scale | projectif | SU2(opt)
    ncols = max(2, sum([True, True, has_combo or has_su2]))
    fig, axes = plt.subplots(2, ncols, figsize=(8 * ncols, 10))
    fig.suptitle('Robustesse — Classic vs Sim(2) vs S²CNN SU(2)',
                 fontsize=14, fontweight='bold')

    # ── Sweep φ ───────────────────────────────────────────────────────────────
    ax = axes[0, 0]
    for tag in tags:
        acc = results[tag]['phi']
        ax.plot(phi_values, acc, marker=MODEL_MARKERS.get(tag, 'o'),
                color=MODEL_COLORS.get(tag, 'gray'), lw=2, ms=5,
                label=f"{MODEL_LABELS.get(tag, tag)}  (Δ={acc[0]-acc[-1]:+.1f}%)")
        ax.axhline(acc[0], color=MODEL_COLORS.get(tag, 'gray'), ls='--', lw=0.8, alpha=0.4)
    ax.set_xlabel('φ (rad)')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Sweep φ — Möbius SU(2)', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # ── Sweep ψ₁ ──────────────────────────────────────────────────────────────
    ax = axes[0, 1]
    for tag in tags:
        acc = results[tag]['psi1']
        ax.plot(psi1_deg, acc, marker=MODEL_MARKERS.get(tag, 'o'),
                color=MODEL_COLORS.get(tag, 'gray'), lw=2, ms=5,
                label=f"{MODEL_LABELS.get(tag, tag)}  (Δ={acc[0]-acc[-1]:+.1f}%)")
        ax.axhline(acc[0], color=MODEL_COLORS.get(tag, 'gray'), ls='--', lw=0.8, alpha=0.4)
    ax.set_xlabel('ψ₁ (°)')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Sweep ψ₁ — rotation planaire Sim(2)', fontsize=11)
    ax.set_xticks([0, 15, 30, 45, 60])
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # ── Combo φ+ψ₁ ────────────────────────────────────────────────────────────
    if has_combo:
        ax = axes[0, 2]
        phi_deg_combo = np.degrees(phi_combo_rad)
        for tag in tags:
            acc = results[tag]['combo']
            ax.plot(phi_deg_combo, acc, marker=MODEL_MARKERS.get(tag, 'o'),
                    color=MODEL_COLORS.get(tag, 'gray'), lw=2, ms=5,
                    label=f"{MODEL_LABELS.get(tag, tag)}  (Δ={acc[0]-acc[-1]:+.1f}%)")
            ax.axhline(acc[0], color=MODEL_COLORS.get(tag, 'gray'), ls='--', lw=0.8, alpha=0.4)
        ax.set_xlabel('φ (°)  [ψ₁ croît simultanément 0→60°]')
        ax.set_ylabel('Accuracy (%)')
        ax.set_title('Sweep Combo φ+ψ₁ — Möbius composé', fontsize=11)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
    elif ncols == 3:
        axes[0, 2].axis('off')

    # ── Sweep Scale ───────────────────────────────────────────────────────────
    ax = axes[1, 0]
    for tag in tags:
        acc = results[tag]['scale']
        ref = float(acc[idx_sc])
        ax.plot(scale_values, acc, marker=MODEL_MARKERS.get(tag, 'o'),
                color=MODEL_COLORS.get(tag, 'gray'), lw=2, ms=6,
                label=f"{MODEL_LABELS.get(tag, tag)}  (@1.0={ref:.1f}%)")
        ax.axhline(ref, color=MODEL_COLORS.get(tag, 'gray'), ls='--', lw=0.8, alpha=0.4)
    ax.axvline(1.0, color='gray', ls=':', lw=1.2, alpha=0.6)
    ax.set_xlabel("Facteur d'échelle")
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Sweep Scale', fontsize=11)
    ax.set_xticks(scale_values)
    ax.set_xticklabels([f'{s:.2f}×' for s in scale_values])
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # ── Projectif (violin) ────────────────────────────────────────────────────
    ax = axes[1, 1]
    data   = [results[t]['proj'] for t in tags]
    labels = [MODEL_LABELS.get(t, t) for t in tags]
    colors = [MODEL_COLORS.get(t, 'gray') for t in tags]
    parts  = ax.violinplot(data, positions=range(len(tags)), showmedians=True)
    for pc, c in zip(parts['bodies'], colors):
        pc.set_facecolor(c)
        pc.set_alpha(0.7)
    ax.set_xticks(range(len(tags)))
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Transformations projectives aléatoires', fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    for ti, (tag, d) in enumerate(zip(tags, data)):
        ax.text(ti, d.mean() + 0.5, f'{d.mean():.1f}%\n±{d.std():.1f}',
                ha='center', va='bottom', fontsize=8,
                color=MODEL_COLORS.get(tag, 'gray'), fontweight='bold')

    # ── SU(2) Haar (violin) ───────────────────────────────────────────────────
    if has_su2:
        ax = axes[1, 2]
        data_su2 = [results[t]['su2'] for t in tags]
        parts    = ax.violinplot(data_su2, positions=range(len(tags)), showmedians=True)
        for pc, c in zip(parts['bodies'], colors):
            pc.set_facecolor(c)
            pc.set_alpha(0.7)
        ax.set_xticks(range(len(tags)))
        ax.set_xticklabels(labels, fontsize=9)
        ax.set_ylabel('Accuracy (%)')
        ax.set_title('Vrai test SU(2) — Haar (6,9 exclus)', fontsize=11)
        ax.grid(True, alpha=0.3, axis='y')
        for ti, (tag, d) in enumerate(zip(tags, data_su2)):
            ax.text(ti, d.mean() + 0.5, f'{d.mean():.1f}%\n±{d.std():.1f}',
                    ha='center', va='bottom', fontsize=8,
                    color=MODEL_COLORS.get(tag, 'gray'), fontweight='bold')
    elif ncols == 3:
        axes[1, 2].axis('off')

    plt.tight_layout()
    path = os.path.join(save_dir, "robustness_curves.png")
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"💾 Robustness curves → {path}")


def save_robustness_tables(rob_results, phi_values_rad, psi1_deg, scale_arr,
                            phi_combo_rad, psi1_combo_rad, save_dir):
    has_combo = phi_combo_rad is not None and 'combo' in next(iter(rob_results.values()))
    has_su2   = 'su2' in next(iter(rob_results.values()))
    tags      = list(rob_results.keys())
    names     = [MODEL_LABELS.get(t, t) for t in tags]

    # ── 1. Sweep φ ────────────────────────────────────────────────────────────
    df_phi = pd.DataFrame({
        'phi_rad': phi_values_rad,
        'phi_deg': np.degrees(phi_values_rad),
        **{names[i]: rob_results[t]['phi'] for i, t in enumerate(tags)}
    })
    df_phi.to_csv(os.path.join(save_dir, "rob_sweep_phi.csv"), index=False)

    # ── 2. Sweep ψ₁ ───────────────────────────────────────────────────────────
    df_psi1 = pd.DataFrame({
        'psi1_deg': psi1_deg,
        **{names[i]: rob_results[t]['psi1'] for i, t in enumerate(tags)}
    })
    df_psi1.to_csv(os.path.join(save_dir, "rob_sweep_psi1.csv"), index=False)

    # ── 3. Sweep Scale ────────────────────────────────────────────────────────
    df_scale = pd.DataFrame({
        'scale': scale_arr,
        **{names[i]: rob_results[t]['scale'] for i, t in enumerate(tags)}
    })
    df_scale.to_csv(os.path.join(save_dir, "rob_sweep_scale.csv"), index=False)

    # ── 4. Sweep Projectif ────────────────────────────────────────────────────
    df_proj = pd.DataFrame({
        'sample_idx': np.arange(len(next(iter(rob_results.values()))['proj'])),
        **{names[i]: rob_results[t]['proj'] for i, t in enumerate(tags)}
    })
    df_proj.to_csv(os.path.join(save_dir, "rob_sweep_proj.csv"), index=False)

    # ── 5. Sweep Combo φ+ψ₁ ──────────────────────────────────────────────────
    if has_combo:
        df_combo = pd.DataFrame({
            'phi_rad':  phi_combo_rad,
            'phi_deg':  np.degrees(phi_combo_rad),
            'psi1_rad': psi1_combo_rad,
            'psi1_deg': np.degrees(psi1_combo_rad),
            **{names[i]: rob_results[t]['combo'] for i, t in enumerate(tags)}
        })
        df_combo.to_csv(os.path.join(save_dir, "rob_sweep_combo.csv"), index=False)

    # ── 6. Vrai test SU(2) Haar ───────────────────────────────────────────────
    if has_su2:
        n_su2 = len(next(iter(rob_results.values()))['su2'])
        df_su2 = pd.DataFrame({
            'sample_idx': np.arange(n_su2),
            **{names[i]: rob_results[t]['su2'] for i, t in enumerate(tags)}
        })
        df_su2.to_csv(os.path.join(save_dir, "rob_sweep_su2.csv"), index=False)

    # ── 7. Résumé global ──────────────────────────────────────────────────────
    idx_sc = int(np.argmin(np.abs(scale_arr - 1.0)))
    rows = []
    for t, name in zip(tags, names):
        pa  = rob_results[t]['phi']
        sa  = rob_results[t]['psi1']
        sca = rob_results[t]['scale']
        pra = rob_results[t]['proj']
        row = {
            'model':       name,
            'phi_at_0':    pa[0],   'phi_drop':    pa[0] - pa[-1],
            'psi1_at_0':   sa[0],   'psi1_drop':   sa[0] - sa[-1],
            'scale_at_1':  sca[idx_sc], 'scale_range': sca.max() - sca.min(),
            'proj_mean':   pra.mean(),  'proj_std':    pra.std(),
        }
        if has_combo:
            ca = rob_results[t]['combo']
            row['combo_at_0'] = ca[0]
            row['combo_drop'] = ca[0] - ca[-1]
        if has_su2:
            su2a = rob_results[t]['su2']
            row['su2_mean'] = su2a.mean()
            row['su2_std']  = su2a.std()
            row['su2_min']  = su2a.min()
        rows.append(row)

    df_summary = pd.DataFrame(rows)
    df_summary.to_csv(os.path.join(save_dir, "rob_summary.csv"), index=False)

    # ── Affichage ─────────────────────────────────────────────────────────────
    fnames = ["rob_sweep_phi.csv", "rob_sweep_psi1.csv", "rob_sweep_scale.csv",
              "rob_sweep_proj.csv"]
    if has_combo: fnames.append("rob_sweep_combo.csv")
    if has_su2:   fnames.append("rob_sweep_su2.csv")
    fnames.append("rob_summary.csv")

    print("\n📊 Robustness tables saved:")
    for fname in fnames:
        print(f"   💾 {os.path.join(save_dir, fname)}")
    print("\n── Résumé ──")
    print(df_summary.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

## 4 · Chargement des modèles pré-entraînés

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CHARGEMENT DES MODÈLES PRÉ-ENTRAÎNÉS  —  boîte noire
#
#  Les modèles sont livrés sérialisés en TorchScript : le notebook n'a besoin
#  d'aucune définition d'architecture. Chaque modèle exporté respecte le contrat
#
#        forward(x : (B,1,64,64))  →  (logits (B,10), features (B,D))
#
#  où `features` est la sortie du pooling global (le même tenseur que celui
#  utilisé pour la mesure d'équivariance dans l'article).
# ══════════════════════════════════════════════════════════════════════════════

def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


class ModelBundle:
    """Enveloppe uniforme autour d'un modèle sérialisé.

    Expose exactement l'interface attendue par les fonctions d'évaluation :
      • bundle(x)          → logits
      • bundle.features(x) → descripteur global aplati (B, D)
      • bundle.eval() / .parameters()
    """

    def __init__(self, tag, net, meta=None):
        self.tag  = tag
        self.net  = net
        self.meta = meta or {}
        self._feat = {}
        # Descripteur = tenseur entrant dans la tête de classification.
        # Uniforme pour les quatre architectures, contrairement à un hook sur
        # le pooling : SU2Stereo n'a pas d'AdaptiveAvgPool2d, et le premier
        # AdaptiveAvgPool2d de BiLogPolar appartient à son estimateur de pôle.
        head = getattr(net, "head", None) or getattr(
            getattr(net, "backbone", None), "head", None)
        if head is None:
            raise RuntimeError(
                f"[{tag}] aucune tête trouvée : impossible d'extraire le "
                f"descripteur.")
        head.register_forward_pre_hook(
            lambda m, inp: self._feat.__setitem__('f', inp[0].detach()))

    def eval(self):
        self.net.eval()
        return self

    def to(self, dev):
        self.net.to(dev)
        return self

    def parameters(self):
        return self.net.parameters()

    def named_modules(self):
        return self.net.named_modules()

    def __call__(self, x):
        out = self.net(x)
        if isinstance(out, (tuple, list)):
            self._feat['f'] = out[1]
            return out[0]
        return out

    def features(self, x):
        self.net(x)
        return self._feat['f'].flatten(1)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def load_manifest(ckpt_dir):
    path = os.path.join(ckpt_dir, "manifest.json")
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"manifest.json introuvable dans {ckpt_dir}. Vérifiez REPO_URL / "
            f"CKPT_SUBDIR dans la cellule de configuration.")
    with open(path) as f:
        return json.load(f)


def load_models(ckpt_dir, manifest, which="trained", verify=True):
    """which ∈ {'trained', 'random'} — 'random' = poids à l'initialisation,
    utilisés pour l'équivariance architecturale (Phase 1 de l'article)."""
    key = "models" if which == "trained" else "random_init"
    entries = manifest.get(key, {})
    bundles = {}
    for tag, info in entries.items():
        fpath = os.path.join(ckpt_dir, info["file"])
        if not os.path.exists(fpath):
            print(f"  ⚠️  [{tag}] fichier absent → ignoré ({info['file']})")
            continue
        if verify and info.get("sha256"):
            got = sha256_of(fpath)
            ok = (got == info["sha256"])
            if not ok:
                print(f"  ❌ [{tag}] SHA-256 non conforme — poids corrompus ou modifiés")
            else:
                print(f"  🔒 [{tag}] SHA-256 vérifié")
        fmt = info.get("format", "torchscript")
        if fmt == "torchscript":
            net = torch.jit.load(fpath, map_location=device)
        else:
            # Repli : le modèle n'a pas pu être sérialisé en TorchScript.
            # Le dépôt fournit alors un module `repro_models` qui reconstruit
            # l'architecture ; le notebook ne l'affiche jamais.
            import importlib, sys as _sys
            if REPO_DIR not in _sys.path:
                _sys.path.insert(0, REPO_DIR)
            rm = importlib.import_module("repro_models")
            net = rm.build(tag, info.get("width_mult"))
            state = torch.load(fpath, map_location=device, weights_only=True)
            state = {k.replace("_orig_mod.", ""): v for k, v in state.items()}
            net.load_state_dict(state)
        net.eval()
        bundles[tag] = ModelBundle(tag, net, info).to(device).eval()
        n_p = count_parameters(bundles[tag])
        ref = info.get("params")
        flag = "✅" if (ref is None or n_p == ref) else f"⚠️ (manifest={ref:,})"
        print(f"  📦 [{tag:<11}] {which:<7} | {n_p:>9,} params {flag}")
    if not bundles and which == "trained":
        raise RuntimeError("Aucun modèle chargé — vérifiez le dossier checkpoints/.")
    return bundles


CKPT_DIR = os.path.join(REPO_DIR, CKPT_SUBDIR) if os.path.isdir(REPO_DIR) else CKPT_SUBDIR
MANIFEST = load_manifest(CKPT_DIR)

print(f"\n📋 Manifeste : {MANIFEST.get('experiment', '?')}")
print(f"   exporté le  : {MANIFEST.get('exported_at', '?')}")
print(f"   torch (export) : {MANIFEST.get('torch_version', '?')}")
print(f"   jeu de données : {MANIFEST.get('dataset', '?')} "
      f"{MANIFEST.get('img_size', '?')}×{MANIFEST.get('img_size', '?')}\n")

trained_models = load_models(CKPT_DIR, MANIFEST, which="trained")
TAGS = list(trained_models.keys())

print()
random_models = load_models(CKPT_DIR, MANIFEST, which="random")
if not random_models:
    print("  ℹ️  Pas de poids à l'initialisation → la Phase 2 utilisera "
          "equivariance_random.json s'il est présent.")

## 5 · Jeu de test

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  DONNÉES  —  MNIST test uniquement (aucun jeu d'entraînement n'est chargé)
#
#  Le pipeline reproduit exactement celui de l'article :
#      Resize(64) → RandomCrop(64, padding=6) → ToTensor → Normalize(0.1307, 0.3081)
#  Le RandomCrop introduit une translation aléatoire de ±6 px, présente aussi
#  à la validation dans le protocole d'origine ; la graine est fixée pour que
#  les chiffres soient reproductibles d'une exécution à l'autre.
# ══════════════════════════════════════════════════════════════════════════════
MEAN, STD = (0.1307,), (0.3081,)
IMG_SIZE  = 64

if EVAL_TRANSFORM == "paper":
    transform_val = transforms.Compose([
        transforms.Resize(64),
        transforms.RandomCrop(64, padding=6),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
elif EVAL_TRANSFORM == "deterministic":
    transform_val = transforms.Compose([
        transforms.Resize(64),
        transforms.Pad(6, padding_mode='edge'),
        transforms.CenterCrop(64),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
else:
    raise ValueError(EVAL_TRANSFORM)

seed_everything()
full_val = datasets.MNIST('./data', train=False, download=True, transform=transform_val)

if CFG['n_val'] is not None and CFG['n_val'] < len(full_val):
    g = torch.Generator().manual_seed(SEED)
    idx = torch.randperm(len(full_val), generator=g)[:CFG['n_val']].tolist()
    val_set = Subset(full_val, sorted(idx))
else:
    val_set = full_val

val_loader = DataLoader(val_set, batch_size=CFG['batch_size'], shuffle=False,
                        num_workers=2, pin_memory=True)

print(f"✅ Jeu de test : {len(val_set)} images "
      f"(sur {len(full_val)}) | transform = « {EVAL_TRANSFORM} »")
print(f"   batch = {CFG['batch_size']} | {len(val_loader)} lots")


@torch.inference_mode()
def evaluate(model, loader, device):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            out = model(x)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return 100. * correct / total

---
## Phase A — Accuracy de référence

Vérifie que les poids livrés reproduisent bien les accuracies publiées. En mode `quick`, un écart de l'ordre du point est attendu (jeu de test sous-échantillonné).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PHASE A — Reproduction des accuracies de référence (Table 1 de l'article)
# ══════════════════════════════════════════════════════════════════════════════
seed_everything()
rows = []
for tag, model in trained_models.items():
    n_p = count_parameters(model)

    t0  = time.time()
    acc = evaluate(model, val_loader, device)
    dt  = time.time() - t0

    # ── Latence : moyenne sur 20 passes après 5 passes de chauffe ────────────
    x_dummy = torch.randn(64, 1, IMG_SIZE, IMG_SIZE, device=device)
    with torch.inference_mode():
        for _ in range(5):
            model(x_dummy)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t1 = time.time()
        for _ in range(20):
            model(x_dummy)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        lat = (time.time() - t1) / 20 / 64 * 1000

    ref = model.meta.get('val_acc')
    delta = None if ref is None else acc - ref
    rows.append({
        'Modèle':        MODEL_LABELS.get(tag, tag),
        'Params':        n_p,
        'Acc (%)':       acc,
        'Acc article':   ref,
        'Δ (pts)':       delta,
        'ms/img':        lat,
        'Éval (s)':      dt,
    })
    print(f"  [{tag:<11}] acc = {acc:6.2f}%"
          + (f"  (article : {ref:.2f}%, Δ = {delta:+.2f})" if ref is not None else "")
          + f"  | {lat:.3f} ms/img")

df_clean = pd.DataFrame(rows)
df_clean.to_csv(os.path.join(OUT_DIR, "table1_clean_accuracy.csv"), index=False)
display(df_clean.style.format({'Params': '{:,.0f}', 'Acc (%)': '{:.2f}',
                               'Acc article': '{:.2f}', 'Δ (pts)': '{:+.2f}',
                               'ms/img': '{:.3f}', 'Éval (s)': '{:.1f}'}))

tol = 0.75 if RUN_MODE == "full" else 2.0
bad = [r for r in rows if r['Δ (pts)'] is not None and abs(r['Δ (pts)']) > tol]
if not bad:
    print(f"\n✅ Tous les modèles sont à ±{tol} pt de la valeur publiée.")
else:
    print(f"\n⚠️  Écart > {tol} pt pour : "
          + ", ".join(r['Modèle'] for r in bad)
          + ("\n   (attendu en mode « quick » : sous-échantillonnage du jeu de test)"
             if RUN_MODE == "quick" else ""))

---
## Phase B — Équivariance

L'erreur cosinus entre le descripteur de `x` et celui de `T·x` mesure à quel point la représentation interne est stable sous la transformation. Mesurée d'abord **à poids aléatoires** — l'équivariance est alors une propriété de l'architecture seule, pas de l'apprentissage — puis sur les modèles entraînés.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PHASE B — Équivariance architecturale (Figure « equivariance » de l'article)
#
#  Deux régimes :
#    • poids aléatoires  → l'équivariance vient de l'architecture seule
#    • poids entraînés   → elle survit à l'optimisation
# ══════════════════════════════════════════════════════════════════════════════
seed_everything()
x_test   = torch.randn(8, 1, 28, 28, device=device)
x_radial = make_radial_images(8, size=28, device=device)

transforms_dict = {
    "Rotation 2D": [
        ("15°", lambda x: apply_rotation_2d(x, 15)),
        ("30°", lambda x: apply_rotation_2d(x, 30)),
        ("45°", lambda x: apply_rotation_2d(x, 45)),
        ("60°", lambda x: apply_rotation_2d(x, 60)),
    ],
    "Möbius φ (θ≠0 réel)": [
        ("φ=6°",  lambda x: apply_mobius_phi(x, math.radians(6))),
        ("φ=13°", lambda x: apply_mobius_phi(x, math.radians(13))),
        ("φ=19°", lambda x: apply_mobius_phi(x, math.radians(19))),
        ("φ=32°", lambda x: apply_mobius_phi(x, math.radians(32))),
        ("φ=45°", lambda x: apply_mobius_phi(x, math.radians(45))),
    ],
    "Möbius φ (θ=0 radial)": [
        ("φ=6°",  lambda x: apply_mobius_phi(x, math.radians(6))),
        ("φ=13°", lambda x: apply_mobius_phi(x, math.radians(13))),
        ("φ=19°", lambda x: apply_mobius_phi(x, math.radians(19))),
        ("φ=32°", lambda x: apply_mobius_phi(x, math.radians(32))),
        ("φ=45°", lambda x: apply_mobius_phi(x, math.radians(45))),
    ],
    "Scale": [
        ("0.7×",  lambda x: apply_scale(x, 0.7)),
        ("0.85×", lambda x: apply_scale(x, 0.85)),
        ("1.2×",  lambda x: apply_scale(x, 1.2)),
        ("1.5×",  lambda x: apply_scale(x, 1.5)),
        ("2.0×",  lambda x: apply_scale(x, 2.0)),
    ],
    "Projectif": [
        ("c=0.001", lambda x: apply_projective(x, 0.001, 0.001)),
        ("c=0.003", lambda x: apply_projective(x, 0.003, 0.003)),
        ("c=0.005", lambda x: apply_projective(x, 0.005, 0.005)),
    ],
    "Möbius φ+ψ₁": [
        ("φ=6°,ψ=15°",  lambda x: apply_mobius_combo(x, math.radians(6),  math.radians(15))),
        ("φ=13°,ψ=30°", lambda x: apply_mobius_combo(x, math.radians(13), math.radians(30))),
        ("φ=19°,ψ=45°", lambda x: apply_mobius_combo(x, math.radians(19), math.radians(45))),
        ("φ=32°,ψ=60°", lambda x: apply_mobius_combo(x, math.radians(32), math.radians(60))),
        ("φ=45°,ψ=60°", lambda x: apply_mobius_combo(x, math.radians(45), math.radians(60))),
    ],
}
x_per_family = {"Möbius φ (θ=0 radial)": x_radial}

# ── B.1 — poids aléatoires ───────────────────────────────────────────────────
eq_random_path = os.path.join(CKPT_DIR, "equivariance_random.json")
if random_models:
    print("── Poids à l'initialisation ──")
    eq_results = measure_equivariance(random_models, x_test, transforms_dict, x_per_family)
elif os.path.exists(eq_random_path):
    print("── Poids à l'initialisation : valeurs relues du dépôt ──")
    with open(eq_random_path) as f:
        raw = json.load(f)
    eq_results = {t: {fam: [(l, e) for l, e in steps] for fam, steps in fams.items()}
                  for t, fams in raw.items()}
else:
    eq_results = None
    print("⚠️  Aucune référence à l'initialisation disponible → B.1 sautée.")

if eq_results is not None:
    print_equivariance_table(eq_results)
    plot_equivariance(eq_results, OUT_DIR, suffix="_random")
    show_png(os.path.join(OUT_DIR, "equivariance_random.png"))
    equivariance_to_dataframe(eq_results).to_csv(
        os.path.join(OUT_DIR, "equivariance_random.csv"), index=False)

# ── B.2 — poids entraînés ────────────────────────────────────────────────────
print("\n── Poids entraînés ──")
eq_trained = measure_equivariance(trained_models, x_test, transforms_dict, x_per_family)
print_equivariance_table(eq_trained)
plot_equivariance(eq_trained, OUT_DIR, suffix="_trained")
show_png(os.path.join(OUT_DIR, "equivariance_trained.png"))
equivariance_to_dataframe(eq_trained).to_csv(
    os.path.join(OUT_DIR, "equivariance_trained.csv"), index=False)

# ── B.3 — figure comparative (aléatoire vs entraîné) ─────────────────────────
if eq_results is not None:
    tags_eq = [t for t in TAGS if t in eq_results and t in eq_trained]
    fig, axes = plt.subplots(2, len(transforms_dict),
                             figsize=(5 * len(transforms_dict), 9), squeeze=False)
    fig.suptitle("Équivariance MNIST : aléatoire (haut) vs entraîné (bas)",
                 fontsize=12, fontweight='bold')
    for col, t_name in enumerate(transforms_dict.keys()):
        for row, (res, title) in enumerate([(eq_results, "Aléatoire"),
                                            (eq_trained, "Entraîné")]):
            ax = axes[row, col]
            n_s = len(res[tags_eq[0]][t_name])
            xpos = np.arange(n_s)
            bw = 0.8 / len(tags_eq)
            xlbl = [l for l, _ in res[tags_eq[0]][t_name]]
            for ti, tag in enumerate(tags_eq):
                errs = [e for _, e in res[tag][t_name]]
                ax.bar(xpos + ti * bw - 0.4 + bw / 2, errs, bw * 0.9,
                       color=MODEL_COLORS.get(tag, 'gray'), alpha=0.85,
                       label=MODEL_LABELS.get(tag, tag), edgecolor='white', lw=0.5)
            ax.axhline(15., color='green', ls='--', lw=1.2, alpha=0.6)
            ax.axhline(40., color='red',   ls='--', lw=1.2, alpha=0.6)
            ax.set_xticks(xpos + 0.4 - bw / 2)
            ax.set_xticklabels(xlbl, fontsize=7, rotation=20, ha='right')
            ax.set_title(f"{t_name} — {title}", fontsize=9)
            ax.set_ylabel('Erreur (%)')
            ax.grid(True, alpha=0.3, axis='y')
            if row == 0 and col == 0:
                ax.legend(fontsize=7)
    plt.tight_layout()
    p = os.path.join(OUT_DIR, "equivariance_comparison.png")
    plt.savefig(p, dpi=150, bbox_inches='tight')
    display(fig)
    plt.close(fig)
    print(f"💾 {p}")

---
## Phase C — Transformations appliquées

Planches de contrôle : elles permettent de vérifier de visu que les amplitudes testées restent sémantiquement valides (un 6 ne devient pas un 9).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PHASE C — Construction des familles de transformations (grilles de sweep)
#
#  φ    : Möbius SU(2) radiale,   0 → 90°
#  ψ₁   : rotation planaire S¹,   0 → 90°
#  Scale: homothéties 0.5× → 3×
#  Proj : homographies aléatoires (graine 42)
#  Combo: φ et ψ₁ croissant conjointement
#  SU(2): tirage selon la mesure de Haar (graine 42), 6↔9 exclus par construction
# ══════════════════════════════════════════════════════════════════════════════
seed_everything()
lin     = torch.linspace(-1, 1, IMG_SIZE, device=device)
grid_r2 = torch.stack(torch.meshgrid(lin, lin, indexing='xy'))

phi_values_rad = np.linspace(0, math.pi / 2, CFG['n_phi'])
phi_grids      = _build_phi_grids_manual(phi_values_rad, grid_r2)

psi1_rad, psi1_deg, psi1_grids = _build_psi1_grids(CFG['n_psi1'], grid_r2,
                                                   phi_fixed=0., max_deg=90.)

SWEEP_SCALE = [0.5, 0.7, 0.85, 1.0, 1.5, 2.0, 3.0]
scale_grids, scale_arr = _build_scale_grids(SWEEP_SCALE, IMG_SIZE, IMG_SIZE, device)

proj_grids = _build_projective_grids(CFG['n_proj'], IMG_SIZE, IMG_SIZE, device,
                                     seed=42, mnist_safe=False)

N_COMBO        = CFG['n_combo']
phi_combo_rad  = np.linspace(0, math.pi / 2, N_COMBO)
psi1_combo_rad = np.linspace(0, math.pi / 2, N_COMBO)
combo_grids    = _build_combo_grids(phi_combo_rad, psi1_combo_rad, grid_r2)

su2_grids, phi_su2, psi1_su2, psi2_su2 = _build_su2_random_grids(
    n_samples = CFG['n_su2'],
    grid_r2   = grid_r2,
    max_phi   = math.pi / 2,
    max_psi1  = math.pi / 2,
    max_psi2  = math.pi / 2,
    seed      = 42,
)

phi_labels   = [f"φ={math.degrees(v):.0f}°" for v in phi_values_rad]
psi1_labels  = [f"ψ₁={d:.0f}°"              for d in psi1_deg]
scale_labels = [f"s={v:.2f}"                for v in SWEEP_SCALE]
proj_labels  = [f"p={i}"                    for i in range(len(proj_grids))]
combo_labels = [f"φ={math.degrees(p):.0f}°\nψ={math.degrees(s):.0f}°"
                for p, s in zip(phi_combo_rad, psi1_combo_rad)]
su2_labels   = [f"φ={math.degrees(phi_su2[i]):.0f}°\n"
                f"ψ₁={math.degrees(psi1_su2[i]):.0f}°\n"
                f"ψ₂={math.degrees(psi2_su2[i]):.0f}°"
                for i in range(len(phi_su2))]

print(f"✅ Grilles : φ={len(phi_grids)} | ψ₁={len(psi1_grids)} | "
      f"scale={len(scale_grids)} | proj={len(proj_grids)} | "
      f"combo={len(combo_grids)} | SU(2)={len(su2_grids)}")
n_fwd = (len(phi_grids) + len(psi1_grids) + len(scale_grids) + len(proj_grids)
         + len(combo_grids) + len(su2_grids))
print(f"   → {n_fwd} passes avant par lot et par modèle "
      f"({n_fwd * len(val_loader) * len(TAGS):,} passes au total)")

# ── Planches visuelles : à quoi ressemblent les images transformées ──────────
VISU_MAX = 16
visualize_transformations(
    val_loader,
    phi_grids    = phi_grids,
    psi1_grids   = psi1_grids,
    scale_grids  = scale_grids,
    proj_grids   = proj_grids[:VISU_MAX],
    combo_grids  = combo_grids,
    su2_grids    = su2_grids[:VISU_MAX],
    phi_labels   = phi_labels,
    psi1_labels  = psi1_labels,
    scale_labels = scale_labels,
    proj_labels  = proj_labels[:VISU_MAX],
    combo_labels = combo_labels,
    su2_labels   = su2_labels[:VISU_MAX],
    save_dir     = OUT_DIR,
    dataset      = 'mnist',
)

---
## Phase D — Sweeps de robustesse

⏱ **Cellule la plus longue.** C'est le résultat central de l'article : accuracy en fonction de l'amplitude de la transformation, pour six familles.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PHASE D — Sweep de robustesse (résultat principal de l'article)
#
#  Pour chaque famille de transformation et chaque amplitude, on mesure
#  l'accuracy top-1 sur le jeu de test. Ce qui compte n'est pas la valeur
#  absolue mais la CHUTE Δ entre l'amplitude 0 et l'amplitude maximale :
#  Δ ≈ 0 signifie invariance.
#
#  ⏱  C'est la cellule la plus longue. Mode « quick » ≈ 3–5 min,
#     mode « full » ≈ 30–45 min sur T4.
# ══════════════════════════════════════════════════════════════════════════════
seed_everything()
t0 = time.time()

rob_results = sweep_all_models(
    trained_models, val_loader,
    phi_grids, psi1_grids, scale_grids, proj_grids,
    combo_grids = combo_grids,
    su2_grids   = su2_grids,
)

print(f"\n⏱  Sweep terminé en {(time.time()-t0)/60:.1f} min")

save_robustness_tables(
    rob_results, phi_values_rad, psi1_deg, scale_arr,
    phi_combo_rad  = phi_combo_rad,
    psi1_combo_rad = psi1_combo_rad,
    save_dir       = OUT_DIR,
)

print_robustness_table(phi_values_rad, psi1_deg, scale_arr, rob_results,
                       phi_combo_rad  = phi_combo_rad,
                       psi1_combo_rad = psi1_combo_rad)

plot_robustness_curves(phi_values_rad, psi1_deg, scale_arr, rob_results, OUT_DIR,
                       phi_combo_rad  = phi_combo_rad,
                       psi1_combo_rad = psi1_combo_rad)
show_png(os.path.join(OUT_DIR, "robustness_curves.png"), width=1400)

# ── Sauvegarde brute pour ré-analyse hors notebook ──────────────────────────
with open(os.path.join(OUT_DIR, "robustness_raw.json"), "w") as f:
    json.dump({t: {k: np.asarray(v).tolist() for k, v in e.items()}
               for t, e in rob_results.items()}, f, indent=2)
print(f"💾 {os.path.join(OUT_DIR, 'robustness_raw.json')}")

---
## Phase E — Courbes d'apprentissage

Relues depuis `history.csv`, enregistré pendant l'entraînement d'origine. Rien n'est ré-entraîné.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PHASE E — Courbes d'apprentissage
#
#  Aucun entraînement n'est relancé : l'historique époque par époque a été
#  enregistré pendant l'entraînement d'origine et est livré avec les poids
#  (history.csv). Cette cellule ne fait que le retracer.
# ══════════════════════════════════════════════════════════════════════════════
hist_path = os.path.join(CKPT_DIR, MANIFEST.get("history", "history.csv"))

if os.path.exists(hist_path):
    df_hist = pd.read_csv(hist_path)
    display(df_hist.head())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for tag, grp in df_hist.groupby('Model'):
        axes[0].plot(grp['Epoch'], grp['Val_Acc'],
                     color=MODEL_COLORS.get(tag, 'gray'),
                     marker=MODEL_MARKERS.get(tag, 'o'), ms=4, lw=2.2,
                     label=MODEL_LABELS.get(tag, tag))
        axes[1].plot(grp['Epoch'], grp['Train_Loss'],
                     color=MODEL_COLORS.get(tag, 'gray'),
                     marker=MODEL_MARKERS.get(tag, 'o'), ms=4, lw=2.2,
                     label=MODEL_LABELS.get(tag, tag))
    axes[0].set_xlabel('Époque'); axes[0].set_ylabel('Accuracy val (%)')
    axes[0].set_title('MNIST — accuracy de validation'); axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[1].set_xlabel('Époque'); axes[1].set_ylabel("Perte d'entraînement")
    axes[1].set_title('MNIST — perte'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    p = os.path.join(OUT_DIR, "training_curves.png")
    plt.savefig(p, dpi=150, bbox_inches='tight')
    display(fig); plt.close(fig)

    best = (df_hist.groupby('Model')['Val_Acc'].max()
            .rename('Meilleure acc. val (%)').reset_index())
    best['Modèle'] = best['Model'].map(lambda t: MODEL_LABELS.get(t, t))
    display(best[['Modèle', 'Meilleure acc. val (%)']])
    df_hist.to_csv(os.path.join(OUT_DIR, "history.csv"), index=False)
else:
    print(f"ℹ️  {hist_path} absent — l'historique d'entraînement n'a pas été "
          f"livré avec les poids. Les autres phases ne sont pas affectées.")

# ── Hyperparamètres d'entraînement déclarés (pour la section « repro » ) ─────
hp_rows = []
for tag, info in MANIFEST.get("models", {}).items():
    hp = info.get("train_hparams", {})
    hp_rows.append({'Modèle': MODEL_LABELS.get(tag, tag),
                    'Époques': info.get('epochs'),
                    'lr': hp.get('lr'), 'weight_decay': hp.get('weight_decay'),
                    'batch': hp.get('batch_size'), 'accum': hp.get('accum'),
                    'width_mult': info.get('width_mult'),
                    'Params': info.get('params')})
if hp_rows:
    df_hp = pd.DataFrame(hp_rows)
    display(df_hp)
    df_hp.to_csv(os.path.join(OUT_DIR, "train_hparams.csv"), index=False)

---
## Phase F — Synthèse

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PHASE F — Récapitulatif et archive des résultats
# ══════════════════════════════════════════════════════════════════════════════
idx_sc = int(np.argmin(np.abs(scale_arr - 1.0)))
summary = []
for tag in TAGS:
    r  = rob_results[tag]
    pa, sa, sca, pra = r['phi'], r['psi1'], r['scale'], r['proj']
    row = {
        'Modèle':   MODEL_LABELS.get(tag, tag),
        'Acc (%)':  float(df_clean.loc[df_clean['Modèle'] == MODEL_LABELS.get(tag, tag),
                                       'Acc (%)'].iloc[0]),
        'ΔΦ':       float(pa[0] - pa[-1]),
        'Δψ₁':      float(sa[0] - sa[-1]),
        'ΔScale':   float(sca.max() - sca.min()),
        'Proj μ':   float(pra.mean()),
    }
    if 'combo' in r:
        row['ΔCombo'] = float(r['combo'][0] - r['combo'][-1])
    if 'su2' in r:
        row['SU(2) μ']   = float(r['su2'].mean())
        row['SU(2) min'] = float(r['su2'].min())
    summary.append(row)

df_final = pd.DataFrame(summary)
df_final.to_csv(os.path.join(OUT_DIR, "reviewer_summary.csv"), index=False)
display(df_final.style.format({c: '{:.2f}' for c in df_final.columns if c != 'Modèle'}))

print("\nLecture : Δ = chute d'accuracy entre amplitude nulle et amplitude maximale.")
print("Δ proche de 0 → invariance ; Δ élevé → le modèle dépend de la pose.")
print("Les classes 1 et 9 sont exclues des sweeps (ambiguïté 6↔9 sous rotation),")
print("comme dans l'article ; les accuracies des sweeps ne sont donc pas")
print("directement comparables à celles de la Phase A.")

# ── Archive ─────────────────────────────────────────────────────────────────
zip_path = os.path.join(OUT_DIR, "reviewer_results.zip")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT_DIR):
        for fn in files:
            if fn.endswith('.zip'):
                continue
            fp = os.path.join(root, fn)
            z.write(fp, os.path.relpath(fp, OUT_DIR))
print(f"\n📦 Archive : {zip_path}")

produced = sorted(f for f in os.listdir(OUT_DIR) if not f.endswith('.zip'))
print(f"\n{len(produced)} fichiers produits :")
for f in produced:
    print(f"   • {f}")

try:
    from google.colab import files as colab_files
    print("\n⬇️  Exécutez  colab_files.download(zip_path)  pour récupérer l'archive.")
except ImportError:
    pass

print(f"\n✅ Reproduction terminée — mode « {RUN_MODE} », transform « {EVAL_TRANSFORM} ».")